### Data Preprocessing for Snow Cover Analysis
**Study area:** Shemonaikha region (50.62°N, 81.92°E)  
**Period:** 2014–2023  
**Data sources:** ERA5-Land, MODIS, SRTM, ground observations

### ERA5-Land: hourly

In [ ]:
// ========================================
// СКАЧИВАНИЕ ERA5-Land ДЛЯ ОДНОГО ПИКСЕЛЯ
// 2014-2023, Daily данные
// ========================================

// Ваша территория (прямоугольник по указанным координатам)
var aoi = ee.Geometry.Polygon([
  [[81.888489, 50.61191], [81.948013, 50.61191], 
   [81.948013, 50.631377], [81.888489, 50.631377], [81.888489, 50.61191]]
]);

// Центральная точка (среднее между координатами)
var point = ee.Geometry.Point([81.918251, 50.6216435]);

// Загружаем данные
var dataset = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
  .filterDate('2014-01-01', '2023-12-31');

// Нужные переменные (все что есть из вашего списка)
var bands = [
  'temperature_2m',
  'u_component_of_wind_10m',
  'v_component_of_wind_10m',
  'total_precipitation_sum',
  'surface_solar_radiation_downwards_sum',
  'surface_thermal_radiation_downwards_sum',
  'forecast_albedo',
  'snow_depth_water_equivalent',
  'snow_cover',
  'snowmelt_sum',
  'snow_density',
  'volumetric_soil_water_layer_1',
  'snow_depth'
];

// Извлекаем значения для точки
var extract = dataset.select(bands).map(function(img) {
  var vals = img.reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: point,
    scale: 11132
  });
  return ee.Feature(null, vals).set('date', img.date().format('YYYY-MM-dd'));
});

// Экспорт
Export.table.toDrive({
  collection: ee.FeatureCollection(extract),
  description: 'era5_2014_2023_daily_region2',
  fileFormat: 'CSV',
  folder: 'GEE_exports'
});

print("✅ ГОТОВО! Перейдите во вкладку TASKS и нажмите RUN");

// Опционально: визуализация точки на карте
Map.centerObject(point, 12);
Map.addLayer(point, {color: 'red'}, 'Точка');
Map.addLayer(aoi, {color: 'blue'}, 'Область');

### MODIS - Snow, Albedo, LST

In [ ]:
// ============================================================
// MODIS ДАННЫЕ С ФИЛЬТРАЦИЕЙ ПО КАЧЕСТВУ
// Продукты: MOD10A1 (снег), MOD11A1 (LST)
// Территория: 50.61191-50.631377, 81.888489-81.948013
// Период: 2014-2023
// ============================================================

// 1. ВАША ТЕРРИТОРИЯ
var aoi = ee.Geometry.Polygon([
  [[81.888489, 50.61191], [81.948013, 50.61191], 
   [81.948013, 50.631377], [81.888489, 50.631377], [81.888489, 50.61191]]
]);

// Центральная точка
var point = ee.Geometry.Point([81.918251, 50.6216435]);

// ============================================================
// 2. MOD10A1 - СНЕЖНЫЙ ПОКРОВ И АЛЬБЕДО (500m)
// ============================================================

var mod10 = ee.ImageCollection('MODIS/061/MOD10A1')
  .filterDate('2014-01-01', '2023-12-31')
  .select(['NDSI_Snow_Cover', 'Snow_Albedo_Daily_Tile', 
           'NDSI_Snow_Cover_Basic_QA', 'NDSI_Snow_Cover_Algorithm_Flags_QA']);

function maskSnowQuality(image) {
  var basicQA = image.select('NDSI_Snow_Cover_Basic_QA');
  var basicMask = basicQA.eq(0);
  
  var flagsQA = image.select('NDSI_Snow_Cover_Algorithm_Flags_QA');
  var inlandMask = flagsQA.bitwiseAnd(1 << 0).eq(0);
  var visibleMask = flagsQA.bitwiseAnd(1 << 1).eq(0);
  var ndsiMask = flagsQA.bitwiseAnd(1 << 2).eq(0);
  var solarMask = flagsQA.bitwiseAnd(1 << 7).eq(0);
  var swirMask = flagsQA.bitwiseAnd(1 << 4).eq(0);
  
  var qualityMask = basicMask.and(inlandMask).and(visibleMask)
                     .and(ndsiMask).and(solarMask).and(swirMask);
  
  return image.updateMask(qualityMask);
}

var snowQualityFiltered = mod10.map(maskSnowQuality);

// ============================================================
// 3. MOD11A1 - LST (Land Surface Temperature, 1km)
// ============================================================

var mod11 = ee.ImageCollection('MODIS/061/MOD11A1')
  .filterDate('2014-01-01', '2023-12-31')
  .select(['LST_Day_1km', 'LST_Night_1km', 'QC_Day', 'QC_Night']);

function maskLSTQuality(image) {
  var qcDay = image.select('QC_Day');
  var dayMask = qcDay.bitwiseAnd(1 << 2).eq(0)
                .and(qcDay.bitwiseAnd(1 << 3).eq(0));
  
  var qcNight = image.select('QC_Night');
  var nightMask = qcNight.bitwiseAnd(1 << 2).eq(0)
                  .and(qcNight.bitwiseAnd(1 << 3).eq(0));
  
  var dayLST = image.select('LST_Day_1km').updateMask(dayMask);
  var nightLST = image.select('LST_Night_1km').updateMask(nightMask);
  
  dayLST = dayLST.multiply(0.02).subtract(273.15).rename('LST_Day_C');
  nightLST = nightLST.multiply(0.02).subtract(273.15).rename('LST_Night_C');
  
  return image.addBands(dayLST).addBands(nightLST)
              .copyProperties(image, ['system:time_start']);
}

var lstQualityFiltered = mod11.map(maskLSTQuality);

// ============================================================
// 4. ФУНКЦИЯ ИЗВЛЕЧЕНИЯ ДАННЫХ ПО ТОЧКЕ
// ============================================================

function extractSnowAndAlbedo(img) {
  var date = img.date();
  var snow = img.select('NDSI_Snow_Cover').reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: point,
    scale: 500,
    bestEffort: true
  });
  var albedo = img.select('Snow_Albedo_Daily_Tile').reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: point,
    scale: 500,
    bestEffort: true
  });
  return ee.Feature(null, {
    'date': date.format('YYYY-MM-dd'),
    'year': date.get('year'),
    'month': date.get('month'),
    'snow_cover': snow.get('NDSI_Snow_Cover'),
    'albedo': albedo.get('Snow_Albedo_Daily_Tile')
  });
}

function extractLST(img) {
  var date = img.date();
  var lstDay = img.select('LST_Day_C').reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: point,
    scale: 1000,
    bestEffort: true
  });
  var lstNight = img.select('LST_Night_C').reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: point,
    scale: 1000,
    bestEffort: true
  });
  return ee.Feature(null, {
    'date': date.format('YYYY-MM-dd'),
    'year': date.get('year'),
    'month': date.get('month'),
    'LST_Day_C': lstDay.get('LST_Day_C'),
    'LST_Night_C': lstNight.get('LST_Night_C')
  });
}

// ============================================================
// 5. ПРИМЕНЯЕМ ФУНКЦИИ
// ============================================================

var snowFeatures = snowQualityFiltered.map(extractSnowAndAlbedo);
var lstFeatures = lstQualityFiltered.map(extractLST);

// ============================================================
// 6. ЭКСПОРТ ОТДЕЛЬНЫХ ФАЙЛОВ
// ============================================================

Export.table.toDrive({
  collection: ee.FeatureCollection(snowFeatures),
  description: 'MODIS_Snow_Albedo_2014_2023_Region2',
  fileFormat: 'CSV',
  folder: 'GEE_exports'
});

Export.table.toDrive({
  collection: ee.FeatureCollection(lstFeatures),
  description: 'MODIS_LST_2014_2023_Region2',
  fileFormat: 'CSV',
  folder: 'GEE_exports'
});

// ============================================================
// 7. ВИЗУАЛИЗАЦИЯ
// ============================================================

Map.centerObject(aoi, 12);
Map.addLayer(aoi, {color: 'red'}, 'Ваша территория');
Map.addLayer(point, {color: 'yellow'}, 'Центральная точка');

// Показываем пример снежного покрова за первый день
var sampleSnow = snowQualityFiltered.first();
if (sampleSnow) {
  Map.addLayer(sampleSnow.select('NDSI_Snow_Cover'), 
    {min: 0, max: 100, palette: ['blue', 'cyan', 'white']}, 
    'Snow Cover (QC filtered)');
}

// Показываем пример LST за первый день
var sampleLST = lstQualityFiltered.first();
if (sampleLST) {
  Map.addLayer(sampleLST.select('LST_Day_C'), 
    {min: -30, max: 30, palette: ['blue', 'green', 'red']}, 
    'LST Day (°C)');
}

print("✅ Скрипт выполнен!");
print("📊 Дней со снежными данными:", snowFeatures.size());
print("📊 Дней с LST данными:", lstFeatures.size());

// Дополнительная информация о территории
print("📍 Координаты AOI:");
print("   Север: 50.631377");
print("   Юг: 50.61191");
print("   Восток: 81.948013");
print("   Запад: 81.888489");
print("📏 Размеры: ~5.3 км × ~2.2 км");

### SRTM

In [ ]:
// ============================================
// SRTM РЕЛЬЕФ ДЛЯ ВАШЕГО УЧАСТКА (ВСЕ 5 ЭКСПОРТОВ)
// Координаты: 50.61191, 81.888489, 50.631377, 81.948013
// ============================================

// 1. ВАШ УЧАСТОК
var region = ee.Geometry.Rectangle({
  coords: [81.888489, 50.61191, 81.948013, 50.631377],
  geodesic: false
});

Map.centerObject(region, 13);
Map.addLayer(region, {color: 'red'}, 'Ваш участок');

// 2. ЗАГРУЗКА SRTM
var srtm = ee.Image('USGS/SRTMGL1_003');
var elevation = srtm.clip(region);

// 3. ПРОИЗВОДНЫЕ
var slope = ee.Terrain.slope(elevation);
var aspect = ee.Terrain.aspect(elevation);

// 4. ЭКСПОЗИЦИЯ (склоны)
var north_facing = aspect.gt(315).or(aspect.lt(45)).rename('north');
var south_facing = aspect.gt(135).and(aspect.lt(225)).rename('south');
var east_facing = aspect.gt(45).and(aspect.lt(135)).rename('east');
var west_facing = aspect.gt(225).and(aspect.lt(315)).rename('west');

// 5. ВИЗУАЛИЗАЦИЯ
Map.addLayer(elevation, {
  min: 200,
  max: 400,
  palette: ['green', 'lightgreen', 'yellow', 'brown', 'gray']
}, 'Высота (м)');

Map.addLayer(slope, {
  min: 0,
  max: 15,
  palette: ['blue', 'green', 'yellow', 'orange', 'red']
}, 'Уклон (градусы)');

Map.addLayer(north_facing.selfMask(), {palette: ['blue']}, 'Северные склоны');
Map.addLayer(south_facing.selfMask(), {palette: ['red']}, 'Южные склоны');
Map.addLayer(east_facing.selfMask(), {palette: ['yellow']}, 'Восточные склоны');
Map.addLayer(west_facing.selfMask(), {palette: ['purple']}, 'Западные склоны');

// 6. СТАТИСТИКА
print('=== СТАТИСТИКА РЕЛЬЕФА ===');

var elev_stats = elevation.reduceRegion({
  reducer: ee.Reducer.minMax().combine(ee.Reducer.mean(), '', true),
  geometry: region,
  scale: 30,
  bestEffort: true
});

print('Минимальная высота (м):', elev_stats.get('elevation_min'));
print('Максимальная высота (м):', elev_stats.get('elevation_max'));
print('Средняя высота (м):', elev_stats.get('elevation_mean'));
print('Перепад высот (м):', 
  ee.Number(elev_stats.get('elevation_max')).subtract(elev_stats.get('elevation_min')));

// Площадь склонов по направлениям
var north_area = north_facing.selfMask()
  .multiply(ee.Image.pixelArea())
  .reduceRegion({
    reducer: ee.Reducer.sum(),
    geometry: region,
    scale: 30,
    maxPixels: 1e9
  });

var south_area = south_facing.selfMask()
  .multiply(ee.Image.pixelArea())
  .reduceRegion({
    reducer: ee.Reducer.sum(),
    geometry: region,
    scale: 30,
    maxPixels: 1e9
  });

var east_area = east_facing.selfMask()
  .multiply(ee.Image.pixelArea())
  .reduceRegion({
    reducer: ee.Reducer.sum(),
    geometry: region,
    scale: 30,
    maxPixels: 1e9
  });

var west_area = west_facing.selfMask()
  .multiply(ee.Image.pixelArea())
  .reduceRegion({
    reducer: ee.Reducer.sum(),
    geometry: region,
    scale: 30,
    maxPixels: 1e9
  });

print('Площадь северных склонов (км²):', 
  ee.Number(north_area.get('north')).divide(1e6));
print('Площадь южных склонов (км²):', 
  ee.Number(south_area.get('south')).divide(1e6));
print('Площадь восточных склонов (км²):', 
  ee.Number(east_area.get('east')).divide(1e6));
print('Площадь западных склонов (км²):', 
  ee.Number(west_area.get('west')).divide(1e6));

// Общая площадь участка
var total_area = region.area().divide(1e6);
print('Общая площадь участка (км²):', total_area);

// ============================================
// 7. ЭКСПОРТЫ (ВСЕ 5)
// ============================================

// 7.1. Высота
Export.image.toDrive({
  image: elevation,
  description: 'Region2_SRTM_elevation',
  folder: 'GEE_Exports',
  fileNamePrefix: 'region2_srtm_elevation',
  scale: 30,
  region: region,
  maxPixels: 1e9,
  fileFormat: 'GeoTIFF'
});

// 7.2. Уклон
Export.image.toDrive({
  image: slope,
  description: 'Region2_SRTM_slope',
  folder: 'GEE_Exports',
  fileNamePrefix: 'region2_srtm_slope',
  scale: 30,
  region: region,
  maxPixels: 1e9,
  fileFormat: 'GeoTIFF'
});

// 7.3. Экспозиция (аспект)
Export.image.toDrive({
  image: aspect,
  description: 'Region2_SRTM_aspect',
  folder: 'GEE_Exports',
  fileNamePrefix: 'region2_srtm_aspect',
  scale: 30,
  region: region,
  maxPixels: 1e9,
  fileFormat: 'GeoTIFF'
});

// 7.4. Северные склоны (маска)
Export.image.toDrive({
  image: north_facing.selfMask(),
  description: 'Region2_SRTM_north_slopes',
  folder: 'GEE_Exports',
  fileNamePrefix: 'region2_srtm_north',
  scale: 30,
  region: region,
  maxPixels: 1e9,
  fileFormat: 'GeoTIFF'
});

// 7.5. Южные склоны (маска)
Export.image.toDrive({
  image: south_facing.selfMask(),
  description: 'Region2_SRTM_south_slopes',
  folder: 'GEE_Exports',
  fileNamePrefix: 'region2_srtm_south',
  scale: 30,
  region: region,
  maxPixels: 1e9,
  fileFormat: 'GeoTIFF'
});

print('\n✅ КОД ГОТОВ!');
print('✅ В панели Tasks справа появится 5 задач:');
print('   1. Region2_SRTM_elevation');
print('   2. Region2_SRTM_slope');
print('   3. Region2_SRTM_aspect');
print('   4. Region2_SRTM_north_slopes');
print('   5. Region2_SRTM_south_slopes');
print('\n✅ Нажмите RUN для каждой задачи');
print('✅ Файлы будут в Google Drive / GEE_Exports');

// Дополнительная информация
print('\n=== ИНФОРМАЦИЯ ОБ УЧАСТКЕ ===');
print('Координаты:');
print('  Север: 50.631377');
print('  Юг: 50.61191');
print('  Восток: 81.948013');
print('  Запад: 81.888489');
print('Размеры: ~5.3 км (Восток-Запад) × ~2.2 км (Север-Юг)');